# This file contains the code for reproducing the experiments of the IEEE 33 bus systems in the paper

This file contains the code for simulating partitioning under varying noise levels (IEEE 33 bus)
For the experiment on congestion levels, refer to ieee_33_bus_cong.ipynb

These files compute coalition 


In [1]:
import networkx as nx
import cvxpy as cp
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from networkx.drawing.nx_pydot import graphviz_layout
from itertools import product
import itertools
from copy import deepcopy
import gc

In [2]:
from time import time

In [3]:
from typing import Any, FrozenSet, Hashable, Iterable, List, Tuple

In [4]:
np.random.seed(111) #Reproducibility

## Define network, battery parameters, scaling, randomization of load profiles

In [6]:
curtailment_cost = 300
voltage_tol = 0.1
imb_cost = 300
bf = 1
num_days = 1000
random_noise = 0.2 #Randomisation of load profiles over nodes.

In [12]:
forecast_noise_levels = [0.0,0.01,0.02,0.05,0.1,0.2]
#In case the above experiment is too long, you can split it into two separate experiments and save data separately first

forecast_noise_levels = [0.0,0.01,0.02,0.05]
forecast_noise_levels = [0.1,0.2]

In [14]:
power_costs_battery_club = 16 #CHF/MWh
power_rating_battery_club_total = 1.6 #MWh (total)
power_costs_battery_residential = 24 #CHF/MWh
power_rating_battery_residential_total = 0.92 #MWh
power_costs_battery_commercial = 8 #CHF/MWh
power_rating_battery_commercial = 6 #MWh

In [16]:
T = 24 #Number of hours in a typical day

## Modified and truncated IEEE 33 bus system (root node as bus 6 in the original)

In [19]:
G = nx.DiGraph()
G.add_nodes_from([0,1,2,3,4,5,6,7,8,9,10,11,12])
G.add_nodes_from([13,14,15,16,17,18,19,20])
G.add_node(21)
G.add_edges_from([(0,1),(1,2),(2,3),(3,4),(4,5),(5,6),(6,7),(7,8),(8,9),(9,10),(10,11),(11,12)])
G.add_edges_from([(0,13),(13,14),(14,15),(15,16),(16,17),(17,18),(18,19),(19,20)])
G.add_edge(0,21)

In [21]:
impedances_long = (1/12.66**2)*np.array([
    [0.19,0.62],
    [1.71, 1.23],
    [1.03, 0.74],
    [1.04, 0.74],
    [0.20, 0.06],
    [0.37, 0.12],
    [1.47, 1.16],
    [0.54, 0.72],
    [0.60, 0.53],
    [0.74, 0.54],
    [1.28, 1.72],
    [0.73, 0.54]
])

impedances_short = (1/12.66**2)*np.array([
    [0.20,0.10],
    [0.28,0.14],
    [1.06,0.95],
    [0.80,0.70],
    [0.51,0.26],
    [0.98,0.96],
    [0.31,0.36],
    [0.34,0.53]
])

powers_long = 1.2*np.array([
    [0.2, 0.1],
    [0.2, 0.1],
    [0.06, 0.02],
    [0.06, 0.02],
    [0.045, 0.03],
    [0.06, 0.035],
    [0.06, 0.035],
    [0.12, 0.08],
    [0.06, 0.01],
    [0.06, 0.02],
    [0.06, 0.02],
    [0.09, 0.04]
])

powers_short = 1.6*np.array([
    [0.06,0.025],
    [0.06,0.025], 
    [0.06, 0.02], 
    [0.12, 0.07], 
    [0.2, 0.6], 
    [0.15, 0.07], 
    [0.21, 0.1], 
    [0.06, 0.04]
]) #Overloaded district

for i in range(len(impedances_long)):
    G.nodes[i+1]['R'] = impedances_long[i,0]
    G.nodes[i+1]['X'] = impedances_long[i,1]
    G.nodes[i+1]['P'] = powers_long[i,0]
    G.nodes[i+1]['Q'] = powers_long[i,1]
    G.nodes[i+1]['alpha_v'] = curtailment_cost*(1/(2*impedances_long[i,0]))
    G.nodes[i+1]['v_tol'] = voltage_tol
    G.nodes[i+1]['power_rating'] = (1+bf)*(3*powers_long[i,0]+powers_long[i,1]) #Battery should be able to absorb PV production which spikes
    G.nodes[i+1]['power_costs'] = power_costs_battery_residential
    G.nodes[i+1]['reactive_costs'] = 0.05*power_costs_battery_residential #Small Regularisation for reactive power for better solutions
    G.nodes[i+1]['type'] = 'residential'

for i in range(len(impedances_short)):
    G.nodes[i+13]['R'] = impedances_short[i,0]
    G.nodes[i+13]['X'] = impedances_short[i,1]
    G.nodes[i+13]['P'] = powers_short[i,0]
    G.nodes[i+13]['Q'] = powers_short[i,1]
    G.nodes[i+13]['alpha_v'] = curtailment_cost*(1/(2*impedances_short[i,0]))
    G.nodes[i+13]['v_tol'] = voltage_tol
    G.nodes[i+13]['power_rating'] = (1+bf)*(powers_short[i,0]+powers_short[i,0]) #Battery should be able to provide peak load which spikes
    G.nodes[i+13]['power_costs'] = power_costs_battery_club
    G.nodes[i+13]['reactive_costs'] = 0.05*power_costs_battery_club #Small Regularisation for reactive power for better solutions
    G.nodes[i+13]['type'] = 'club'

G.nodes[21]['R'] = 0
G.nodes[21]['X'] = 0
G.nodes[21]['P'] = 1
G.nodes[21]['Q'] = 0
G.nodes[21]['alpha_v'] = 1
G.nodes[21]['v_tol'] = voltage_tol
G.nodes[21]['power_rating'] = (1+bf)*deepcopy(G.nodes[21]['P']+np.sum(np.abs(powers_short)))
G.nodes[21]['power_costs'] = power_costs_battery_commercial
G.nodes[21]['reactive_costs'] = 0.05*power_costs_battery_commercial #Small Regularisation for reactive power for better solutions
G.nodes[21]['type'] = 'commercial'

## Helper functions to create a subgraph of the original network corresponding to coalitions

In [24]:
def extract_type_subgraph_with_root(G, types_to_keep):
    # Detect root (in-degree 0)
    roots = [n for n in G.nodes if G.in_degree(n) == 0]
    if not roots:
        raise ValueError("No root found (node with in-degree 0)")
    root = roots[0]

    # Nodes to keep
    selected_nodes = [n for n, d in G.nodes(data=True) if d.get('type') in types_to_keep]
    
    # Optionally include the root
    if root not in selected_nodes:
        selected_nodes.append(root)
    
    # Induce subgraph
    subG = G.subgraph(selected_nodes).copy()

    return subG

## Helper functions for partitions and coalitions

In [27]:
class CoalitionLattice:
    def __init__(self, entries: Iterable[Hashable]):
        self.entries: List[Any] = list(entries)
        self.n: int = len(self.entries)
        self._bit = {e: k for k, e in enumerate(self.entries)}
        if len(self._bit) != self.n:
            raise ValueError("entries must be unique and hashable")
 
        # ---- 1. all non-empty coalitions, indexed from 0 ----
        #   list index i  <->  bitmask (i + 1)
        self.coalitions: List[List[Any]] = [
            [self.entries[k] for k in range(self.n) if (mask >> k) & 1]
            for mask in range(1, 1 << self.n)
        ]
 
        # ---- 2. all partitions, indexed from 0 ----
        self.partitions: List[List[List[Any]]] = []
        self.partition_block_indices: List[FrozenSet[int]] = []
        self._partition_index: Dict[FrozenSet[int], int] = {}
 
        for raw in self._iter_set_partitions(list(range(self.n))):
            # each raw block -> bitmask (>= 1) -> 0-based coalition index (mask - 1)
            block_indices = sorted(
                sum(1 << k for k in block) - 1 for block in raw
            )
            partition = [self.coalitions[i] for i in block_indices]
            key = frozenset(block_indices)
            self._partition_index[key] = len(self.partitions)
            self.partition_block_indices.append(key)
            self.partitions.append(partition)
 
    # ---------------- coalitions ----------------
    def coalition(self, index: int) -> List[Any]:
        """Coalition (list of entries) for a given index (0 .. 2**n - 2)."""
        return self.coalitions[index]
 
    def coalition_index(self, coalition: Iterable[Hashable]) -> int:
        """Index for a given coalition (accepts any non-empty iterable of entries)."""
        mask = 0
        for e in coalition:
            mask |= 1 << self._bit[e]
        if mask == 0:
            raise ValueError("the empty coalition has no index")
        return mask - 1
 
    # ---------------- partitions ----------------
    def partition(self, index: int) -> List[List[Any]]:
        """Partition (list of coalitions) for a given index."""
        return self.partitions[index]
 
    def partition_index(self, partition: Iterable[Iterable[Hashable]]) -> int:
        """Index for a given partition (accepts any iterable of blocks)."""
        return self._partition_index[
            frozenset(self.coalition_index(block) for block in partition)
        ]
 
    def coalition_indices(self, partition_index: int) -> FrozenSet[int]:
        """The set of coalition indices that make up a partition."""
        return self.partition_block_indices[partition_index]
 
    def from_coalition_indices(
        self, indices: Iterable[int]
    ) -> Tuple[int, List[List[Any]]]:
        """(partition index, partition) for a given set of coalition indices."""
        j = self._partition_index[frozenset(indices)]
        return j, self.partitions[j]
 
    # ---------------- helpers ----------------
    @property
    def num_coalitions(self) -> int:
        return len(self.coalitions)          # 2**n - 1
 
    @property
    def num_partitions(self) -> int:
        return len(self.partitions)          # Bell number B(n)
 
    @staticmethod
    def _iter_set_partitions(items):
        """Yield every set partition of `items` as a list of blocks."""
        if len(items) <= 1:
            yield [list(items)] if items else []
            return
        first, rest = items[0], items[1:]
        for smaller in CoalitionLattice._iter_set_partitions(rest):
            for i, block in enumerate(smaller):
                yield smaller[:i] + [[first] + block] + smaller[i + 1:]
            yield [[first]] + smaller

In [29]:
C_L = CoalitionLattice(['club', 'residential', 'commercial'])

In [31]:
C_L.coalitions

[['club'],
 ['residential'],
 ['club', 'residential'],
 ['commercial'],
 ['club', 'commercial'],
 ['residential', 'commercial'],
 ['club', 'residential', 'commercial']]

In [33]:
C_L.partitions

[[['club', 'residential', 'commercial']],
 [['club'], ['residential', 'commercial']],
 [['club', 'residential'], ['commercial']],
 [['residential'], ['club', 'commercial']],
 [['club'], ['residential'], ['commercial']]]

In [35]:
Phi_partition = np.zeros((len(forecast_noise_levels), len(C_L.partitions)))
phi_coalition = np.zeros((len(forecast_noise_levels), len(C_L.coalitions), len(C_L.partitions)))
phi0_coalition = np.zeros((len(forecast_noise_levels), len(C_L.coalitions)))

## Functions to get ex-ante and ex-post costs for a coalition

In [38]:
def opf_coalition(subG, day_index):
    
    nodes_in_graph = list(subG.nodes)
    nodes_in_graph_load = nodes_in_graph.copy()
    nodes_in_graph_load.remove(0)
    
    #Need mapping from coalition nodes to index within coalition
    #Need mapping from index within coalition to coalition nodes
    nodes_to_index = {0:0}
    index_to_nodes = {0:0}
    
    for i in range(len(nodes_in_graph_load)):
        nodes_to_index[nodes_in_graph_load[i]] = i+1
        index_to_nodes[i+1] = nodes_in_graph_load[i]
    
    x_P = cp.Variable((len(nodes_in_graph_load),T)) #Number of prosumers times timesteps
    x_Q = cp.Variable((len(nodes_in_graph_load),T))
    epi_x_P = cp.Variable((len(nodes_in_graph_load),T)) #Number of prosumers times timesteps
    epi_x_Q = cp.Variable((len(nodes_in_graph_load),T))

    P_nodes = cp.Variable((len(nodes_in_graph_load)+1,T))
    Q_nodes = cp.Variable((len(nodes_in_graph_load)+1,T))

    del_v_nodes = cp.Variable((len(nodes_in_graph_load)+1,T))
    
    constraints = []

    active_costs_prosumers = np.zeros((1,len(nodes_in_graph_load)))
    reactive_costs_prosumers = np.zeros((1,len(nodes_in_graph_load)))
    
    #Write flow constraints for prosumer nodes
    for i in range(len(nodes_in_graph_load)):
        n = nodes_in_graph_load[i]

        active_costs_prosumers[0,nodes_to_index[n]-1] = subG.nodes[n]['power_costs']
        reactive_costs_prosumers[0,nodes_to_index[n]-1] = subG.nodes[n]['reactive_costs']
        
        prosumer_P_forecasts = subG.nodes[n]['hat_X_P'][day_index]
        prosumer_Q_forecasts = subG.nodes[n]['hat_X_Q'][day_index]
        prosumer_battery_capacity = subG.nodes[n]['power_rating']
        
        descendent_nodes = list(subG.successors(n))
        if len(descendent_nodes)>0:
            descendent_node_idxs = np.zeros(len(descendent_nodes),dtype=np.uint8)
            
            for j in range(len(descendent_nodes)):
                descendent_node_idxs[j] = nodes_to_index[descendent_nodes[j]]
            
            for t in range(T):
                #Power balances
                constraints.append(P_nodes[nodes_to_index[n],t]-cp.sum(P_nodes[descendent_node_idxs,t])
                                   -prosumer_P_forecasts[t]+x_P[nodes_to_index[n]-1,t]==0)
                constraints.append(Q_nodes[nodes_to_index[n],t]-cp.sum(Q_nodes[descendent_node_idxs,t])
                                   -prosumer_Q_forecasts[t]+x_Q[nodes_to_index[n]-1,t]==0)

                #Feasible battery region
                constraints.append(epi_x_P[nodes_to_index[n]-1,t]+epi_x_Q[nodes_to_index[n]-1,t]-prosumer_battery_capacity<=0)
                
                #Epigraphical constraints for |x_P|, |x_Q|
                
                constraints.append(x_P[nodes_to_index[n]-1,t]-epi_x_P[nodes_to_index[n]-1,t]<=0)
                constraints.append(-x_P[nodes_to_index[n]-1,t]-epi_x_P[nodes_to_index[n]-1,t]<=0)
                constraints.append(x_Q[nodes_to_index[n]-1,t]-epi_x_Q[nodes_to_index[n]-1,t]<=0)
                constraints.append(-x_Q[nodes_to_index[n]-1,t]-epi_x_Q[nodes_to_index[n]-1,t]<=0)
                constraints.append(-epi_x_P[nodes_to_index[n]-1,t]<=0)
                constraints.append(-epi_x_Q[nodes_to_index[n]-1,t]<=0)
                
                
        else:
            for t in range(T):
                constraints.append(P_nodes[nodes_to_index[n],t]-prosumer_P_forecasts[t]+x_P[nodes_to_index[n]-1,t]==0)
                constraints.append(Q_nodes[nodes_to_index[n],t]-prosumer_Q_forecasts[t]+x_Q[nodes_to_index[n]-1,t]==0)
                
                #Feasible battery region
                constraints.append(epi_x_P[nodes_to_index[n]-1,t]+epi_x_Q[nodes_to_index[n]-1,t]-prosumer_battery_capacity<=0)
                
                #Epigraphical constraints for |x_P|, |x_Q|
                
                constraints.append(x_P[nodes_to_index[n]-1,t]-epi_x_P[nodes_to_index[n]-1,t]<=0)
                constraints.append(-x_P[nodes_to_index[n]-1,t]-epi_x_P[nodes_to_index[n]-1,t]<=0)
                constraints.append(x_Q[nodes_to_index[n]-1,t]-epi_x_Q[nodes_to_index[n]-1,t]<=0)
                constraints.append(-x_Q[nodes_to_index[n]-1,t]-epi_x_Q[nodes_to_index[n]-1,t]<=0)
                constraints.append(-epi_x_P[nodes_to_index[n]-1,t]<=0)
                constraints.append(-epi_x_Q[nodes_to_index[n]-1,t]<=0)
                
    #Write flow constraints for root node
    descendent_nodes_root = list(subG.successors(0))
    descendent_node_idxs = np.zeros(len(descendent_nodes_root),dtype=np.uint8)
    for i in range(len(descendent_nodes_root)):
        descendent_node_idxs[i] = nodes_to_index[descendent_nodes_root[i]]
    for t in range(T):
        constraints.append(P_nodes[0,t]-cp.sum(P_nodes[descendent_node_idxs,t])==0)
        constraints.append(Q_nodes[0,t]-cp.sum(Q_nodes[descendent_node_idxs,t])==0)
    

    #Write voltage constraints for prosumer nodes
    for i in range(len(nodes_in_graph_load)):
        n = nodes_in_graph_load[i]
        a_n = list(G.predecessors(n))[0]

        resis = G.nodes[n]['R']
        react = G.nodes[n]['X']
        
        n_idx = nodes_to_index[n]
        a_n_idx = nodes_to_index[a_n]

        for t in range(T):
            constraints.append(del_v_nodes[a_n_idx,t]-del_v_nodes[n_idx, t]+2*(resis*P_nodes[n_idx,t]+react*Q_nodes[n_idx,t])==0)
    
    #Write voltage limit constraints
    for i in range(len(nodes_in_graph)):
        n = nodes_in_graph[i]
        n_idx = nodes_to_index[n]
        
        for t in range(T):
            constraints.append(del_v_nodes[n_idx,t]-voltage_tol<=0)
            constraints.append(-del_v_nodes[n_idx,t]-voltage_tol<=0)
    
    #Write self consumption constraint
    for t in range(T):
        constraints.append(P_nodes[0,t]==0)
        constraints.append(Q_nodes[0,t]==0)

    #Write root voltage constraint
    for t in range(T):
        constraints.append(del_v_nodes[0,t]==0)
    
    #Write objective function
    obj_coalition = cp.sum(active_costs_prosumers@epi_x_P+reactive_costs_prosumers@epi_x_Q)
    
    #Solve cvxpy and get costs and variables
    prob = cp.Problem(cp.Minimize(obj_coalition), constraints)
    opf_cost = prob.solve()
    
    x_P_sol = x_P.value
    x_Q_sol = x_Q.value
    
    P_nodes_sol = P_nodes.value
    Q_nodes_sol = Q_nodes.value
    
    del_v_nodes_sol = del_v_nodes.value

    #opf_cost = prob.solve()

    # Explicit cleanup to avoid memory leak
    prob = None
    constraints.clear()
    
    del x_P, x_Q, epi_x_P, epi_x_Q
    del P_nodes, Q_nodes, del_v_nodes
    del constraints

    gc.collect()
    
    return opf_cost, x_P_sol, x_Q_sol, P_nodes_sol, Q_nodes_sol, del_v_nodes_sol, nodes_to_index, index_to_nodes

In [40]:
def ex_post_costs_coalition(subG, x_P_sol, x_Q_sol, nodes_to_index, index_to_nodes, day_index):
    
    voltage_costs = np.zeros((len(nodes_to_index), T))
    balancing_costs = np.zeros(T)

    P_nodes = np.zeros((len(nodes_to_index), T))
    Q_nodes = np.zeros_like(P_nodes)
    del_v_nodes = np.zeros_like(P_nodes)

    #Calculate branch power flows
    for i in range(1, len(nodes_to_index)):
        n = index_to_nodes[i]
        n_ = deepcopy(n)
        prosumer_P_realizations = subG.nodes[n]['X_P'][day_index]
        prosumer_Q_realizations = subG.nodes[n]['X_Q'][day_index]
                
        for t in range(T):
            P_nodes[i,t] = (prosumer_P_realizations[t]-x_P_sol[i-1,t]).copy()
            Q_nodes[i,t] = (prosumer_Q_realizations[t]-x_Q_sol[i-1,t]).copy()

        while(n_!=0):
            ancestor_node = list(subG.predecessors(n_))[0]
            
            P_nodes[nodes_to_index[ancestor_node]]+=P_nodes[i].copy()
            Q_nodes[nodes_to_index[ancestor_node]]+=Q_nodes[i].copy()
            n_ = ancestor_node

    #Calculate voltages
    for i in range(len(nodes_to_index)):
        n = index_to_nodes[i]
        descendents = list(subG.successors(n))
        for d in descendents:
            d_i = nodes_to_index[d]
            del_v_nodes[d_i] = del_v_nodes[i]+2*(subG.nodes[d]['R']*P_nodes[d_i]+subG.nodes[d]['X']*Q_nodes[d_i])
    
    #Write voltage limit costs
    for n in list(subG.nodes):
        if n!=0:
            n_idx = nodes_to_index[n]
            voltage_cost_factor = subG.nodes[n]['alpha_v']
            for t in range(T):
                if np.abs(del_v_nodes[n_idx,t])>voltage_tol:
                    voltage_costs[n_idx,t] = voltage_cost_factor*(np.abs(del_v_nodes[n_idx,t])-voltage_tol)

    #Write balancing costs
    for t in range(T):
        balancing_costs[t] = imb_cost*np.sqrt(P_nodes[0,t]**2+Q_nodes[0,t]**2)
    
    vol_costs = np.sum(voltage_costs)
    bal_costs = np.sum(balancing_costs)

    return vol_costs, bal_costs, P_nodes, Q_nodes, del_v_nodes

## Defining the load and generation profiles, forecasts and realisations for each of the branches

### Branch: Nightclub consumption+wind generation

In [44]:
consumption_template_club = np.roll(np.append(np.array([0.1,0.15,0.15,0.1]),np.zeros(20)),-1)
consumption_template_club = np.where(consumption_template_club>0, consumption_template_club,
                                         consumption_template_club+(1-np.sum(consumption_template_club))/(T-len(np.where(consumption_template_club>0)[0])))

In [46]:
scaling_club = np.max(consumption_template_club)

In [48]:
num_consumers = len(impedances_short)

consumptions_club = np.zeros((num_consumers, num_days, T))
for i in range(num_consumers):
    consumption_random = (1+random_noise*np.random.rand(T))*(consumption_template_club)
    for j in range(num_days):
        roll_idx = np.where(np.random.multinomial(1, [0.25,0.5, 0.25]))[0][0]-1
        cons_node = np.roll(consumption_random, roll_idx)
        consumptions_club[i,j] = cons_node

mean_rayleigh = 1/T
scale = mean_rayleigh/1.253 #Rayleigh scaling

productions_club = np.zeros((num_consumers, num_days, T))
wind = np.random.rayleigh(scale, size=T)
for i in range(num_days):
    for j in range(num_consumers):
        productions_club[j,i] = wind

In [50]:
def generate_stochastic_prosumptions_club(prods, cons, noise_factor):
    n_cons, n_days, T = prods.shape
    prod_forecasts = np.zeros_like(prods)
    cons_forecasts = np.zeros_like(cons)

    #Cons forecasts
    for i in range(n_cons):
        for j in range(n_days):
            for k in range(T):
                cons_forecasts[i,j,k] = ((1+noise_factor*(2*np.random.rand()-1))*cons[i,j,k]).copy()
    
    #Prod forecasts
    for j in range(n_days):
        for k in range(T):
            wind = prods[0,j,k]
            prod_forecasts[:,j,k] = (1+noise_factor*(2*np.random.rand()-1))*wind*np.ones(n_cons)

    return prod_forecasts, cons_forecasts

### Branch: Residential+PV

In [53]:
#Peak at 7am 8am and 8pm 9pm consumption
#Production bell curve: 
consumption_template_residential = np.array([25,20,20,20,22,22,25,30,35,35,33,31,30,27,27,25,25,27,30,38,42,42,38,30.0])
consumption_template_residential/= np.sum(consumption_template_residential)
pv_production_template = np.array([0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.01, 0.26, 1.85, 2.73, 5.81, 7.92, 9.51, 10.08, 9.75, 9.16, 7.33, 3.77, 1.71, 0.68, 0.07, 0.00, 0.00, 0.00])
pv_production_template/=np.sum(pv_production_template)

In [55]:
scaling_residential = np.max(consumption_template_residential)

In [57]:
num_consumers = len(impedances_long)

consumptions_residential = np.zeros((num_consumers, num_days, T))
for i in range(num_consumers):
    consumption_random = (1+random_noise*np.random.rand(T))*(consumption_template_residential)
    for j in range(num_days):
        roll_idx = np.where(np.random.multinomial(1, [0.25,0.5, 0.25]))[0][0]-1
        cons_node = np.roll(consumption_random, roll_idx)
        consumptions_residential[i,j] = cons_node

productions_residential = np.zeros((num_consumers, num_days, T))
solar = (1+random_noise*np.random.rand(T))*pv_production_template
for i in range(num_days):
    
    for j in range(num_consumers):
        productions_residential[j,i] = solar

In [59]:
def generate_stochastic_prosumptions_residential(prods, cons, noise_factor):
    n_cons, n_days, T = prods.shape
    prod_forecasts = np.zeros_like(prods)
    cons_forecasts = np.zeros_like(cons)

    #Cons forecasts
    for i in range(n_cons):
        for j in range(n_days):
            for k in range(T):
                cons_forecasts[i,j,k] = ((1+noise_factor*(2*np.random.rand()-1))*cons[i,j,k]).copy()
    
    #Prod forecasts
    for j in range(n_days):
        for k in range(T):
            solar = prods[0,j,k]
            prod_forecasts[:,j,k] = (1+noise_factor*(2*np.random.rand()-1))*solar*np.ones(n_cons)

    return prod_forecasts, cons_forecasts

### Branch: Commercial consumption profile, constant generation (waste incinerator)

In [62]:
# On time main: 9am to 5pm, with 1 hour buffer

base_consumption_template_com = np.array([0.03, 0.06, 0.08, 0.09, 0.09, 0.09, 0.09, 0.09, 0.09, 0.08, 0.06, 0.03])
base_consumption_template_com = np.array([10,10,10,10,10,10,10,12,15,18,20,20,20,20,20,20,20,20,15,12,10,10,10.0,10])
base_consumption_template_com/=np.sum(base_consumption_template_com)
consumption_template_com = base_consumption_template_com.copy()

#consumption_template_com[7:19] = base_consumption_template_com
#baseline_consumption_com = (1-np.sum(base_consumption_template_com))/(T-len(base_consumption_template_com))

#consumption_template_com[0:7] = baseline_consumption_com
#consumption_template_com[19:] = baseline_consumption_com

production_template_waste = (1/T)*np.ones(T)

In [64]:
num_consumers = 1

consumptions_commercial = np.zeros((num_consumers, num_days, T))
productions_commercial = np.zeros_like(consumptions_commercial)
for i in range(num_consumers):
    for j in range(num_days):
        consumption_random = (1+random_noise*np.random.rand(T))*(consumption_template_com)
        consumptions_commercial[i,j] = consumption_random

for i in range(num_days):
    for j in range(num_consumers):
        productions_commercial[j,i] = production_template_waste

## Simulate and save data

In [67]:
total_costs_coalitions = np.zeros((len(forecast_noise_levels),len(C_L.coalitions),num_days))
opf_costs_coalitions = np.zeros_like(total_costs_coalitions)
balancing_costs_coalitions = np.zeros_like(total_costs_coalitions)
voltage_costs_coalitions = np.zeros_like(total_costs_coalitions)

tbeg = time()
for noise_i in range(len(forecast_noise_levels)):
    print(forecast_noise_levels[noise_i])
    prod_forecasts_club, cons_forecasts_club = generate_stochastic_prosumptions_club(productions_club, 
                                                                                              consumptions_club, 
                                                                                              noise_factor=forecast_noise_levels[noise_i])
    
    prod_forecasts_residential, cons_forecasts_residential = generate_stochastic_prosumptions_residential(productions_residential, 
                                                                                              consumptions_residential, 
                                                                                              noise_factor=forecast_noise_levels[noise_i])
    for i in range(1,13):
        p_scaling = G.nodes[i]['P']
        q_scaling = G.nodes[i]['Q']
        
        prods_P_scaled = (p_scaling/scaling_residential)*(productions_residential[i-1]).copy()
        cons_P_scaled = (p_scaling/scaling_residential)*(consumptions_residential[i-1]).copy()
        prods_P_f_scaled = (p_scaling/scaling_residential)*(prod_forecasts_residential[i-1]).copy()
        cons_P_f_scaled = (p_scaling/scaling_residential)*(cons_forecasts_residential[i-1]).copy()
        cons_Q_scaled = (q_scaling/scaling_residential)*(consumptions_residential[i-1]).copy()
        cons_Q_f_scaled = (q_scaling/scaling_residential)*(cons_forecasts_residential[i-1]).copy()
        
        G.nodes[i]['X_P'] = prods_P_scaled-cons_P_scaled
        G.nodes[i]['hat_X_P'] = prods_P_f_scaled-cons_P_f_scaled
        G.nodes[i]['X_Q'] = -cons_Q_scaled
        G.nodes[i]['hat_X_Q'] = -cons_Q_f_scaled
        
    
    #club
    for i in range(13,21):
        p_scaling = G.nodes[i]['P']
        q_scaling = G.nodes[i]['Q']
        
        prods_P_scaled = (p_scaling/scaling_club)*(productions_club[i-13]).copy()
        cons_P_scaled = (p_scaling/scaling_club)*(consumptions_club[i-13]).copy()
        prods_P_f_scaled = (p_scaling/scaling_club)*(prod_forecasts_club[i-13]).copy()
        cons_P_f_scaled = (p_scaling/scaling_club)*(cons_forecasts_club[i-13]).copy()
        cons_Q_scaled = (q_scaling/scaling_club)*(consumptions_club[i-13]).copy()
        cons_Q_f_scaled = (q_scaling/scaling_club)*(cons_forecasts_club[i-13]).copy()
        
        G.nodes[i]['X_P'] = prods_P_scaled-cons_P_scaled
        G.nodes[i]['hat_X_P'] = prods_P_f_scaled-cons_P_f_scaled
        G.nodes[i]['X_Q'] = -cons_Q_scaled
        G.nodes[i]['hat_X_Q'] = -cons_Q_f_scaled
        
    #commercial
    G.nodes[21]['X_P'] = (productions_commercial[0]-consumptions_commercial[0]).copy()
    G.nodes[21]['hat_X_P'] = (productions_commercial[0]-consumptions_commercial[0]).copy() #No forecast error for this
    G.nodes[21]['X_Q'] = np.zeros_like(consumptions_commercial[0])
    G.nodes[21]['hat_X_Q'] = np.zeros_like(consumptions_commercial[0])

    subGs = [None for k in range(len(C_L.coalitions))]

    for j in range(len(subGs)):
        subGs[j] = (extract_type_subgraph_with_root(G, C_L.coalitions[j]))

    
    for j in range(len(C_L.coalitions)):
        print(C_L.coalitions[j])
        print('===========')
        for k in tqdm(range(num_days)):
            opf_cost, x_P_sol, x_Q_sol, P_nodes_sol, Q_nodes_sol, del_v_nodes_sol, nodes_to_index, index_to_nodes = opf_coalition(subGs[j], k)
            vol_costs, bal_costs, P_nodes, Q_nodes, del_v_nodes = ex_post_costs_coalition(subGs[j], x_P_sol, x_Q_sol, nodes_to_index, index_to_nodes, k)

            #print(del_v_nodes_sol)
            #print(vol_costs)
            #print(bal_costs)
            #print(opf_cost)
            
            total_costs_coalitions[noise_i,j,k] = opf_cost+vol_costs+bal_costs
            opf_costs_coalitions[noise_i,j,k] = opf_cost
            balancing_costs_coalitions[noise_i,j,k] = bal_costs
            voltage_costs_coalitions[noise_i,j,k] = vol_costs
            
            
            np.savez('experiments2_29jun26'+str(noise_i)+'.npz',total=total_costs_coalitions,opf=opf_costs_coalitions,bal=balancing_costs_coalitions,vol=voltage_costs_coalitions, noise=forecast_noise_levels)

        phi0_coalition[noise_i,j] = np.mean(total_costs_coalitions[noise_i,j])

    for p_idx in range(len(C_L.partitions)):
        part = C_L.partitions[p_idx]
        part_cost = 0
        for c_idx in range(len(part)):
            coal = part[c_idx]
            coal_index = C_L.coalition_index(coal)
            coal_cost = phi0_coalition[noise_i,coal_index]

            part_cost+=coal_cost.copy()
            phi_coalition[noise_i,coal_index,p_idx] = coal_cost.copy()

        Phi_partition[noise_i, p_idx] = part_cost.copy()

    np.savez('experiments2_total_costs_29jun26'+str(noise_i)+'.npz',Phi_partition=Phi_partition, phi_coalition=phi_coalition, C_L=np.array(C_L, dtype=object))

tend = time()

0.1
['club']


100%|███████████████████████████████████████| 1000/1000 [15:50<00:00,  1.05it/s]


['residential']


100%|███████████████████████████████████████| 1000/1000 [23:53<00:00,  1.43s/it]


['club', 'residential']


...

100%|███████████████████████████████████████| 1000/1000 [16:56<00:00,  1.02s/it]


['residential', 'commercial']


100%|███████████████████████████████████████| 1000/1000 [25:23<00:00,  1.52s/it]


['club', 'residential', 'commercial']


100%|███████████████████████████████████████| 1000/1000 [44:05<00:00,  2.65s/it]


In [68]:
total_time = tend-tbeg
print(total_time)

20594.89075398445


In [70]:
np.mean(total_costs_coalitions[-1,:,:],axis=-1)

array([1.45449730e+01, 3.88373237e+01, 8.01468466e+01, 6.42868838e-02,
       1.68128491e+01, 6.77166210e+01, 7.76908441e+01])

In [75]:
Phi_partition[-1]

array([187.49823998, 218.74762695, 194.18686367, 157.73914504,
       159.49866643])

In [77]:
C_L.partitions

[[['club', 'residential', 'commercial']],
 [['club'], ['residential', 'commercial']],
 [['club', 'residential'], ['commercial']],
 [['residential'], ['club', 'commercial']],
 [['club'], ['residential'], ['commercial']]]

In [79]:
phi_coalition[-1]

array([[0.00000000e+00, 5.13533696e+01, 0.00000000e+00, 0.00000000e+00,
        5.13533696e+01],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 1.08080435e+02,
        1.08080435e+02],
       [0.00000000e+00, 0.00000000e+00, 1.94122002e+02, 0.00000000e+00,
        0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 6.48614567e-02, 0.00000000e+00,
        6.48614567e-02],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 4.96587097e+01,
        0.00000000e+00],
       [0.00000000e+00, 1.67394257e+02, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00],
       [1.87498240e+02, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00]])

In [81]:
C_L.coalitions

[['club'],
 ['residential'],
 ['club', 'residential'],
 ['commercial'],
 ['club', 'commercial'],
 ['residential', 'commercial'],
 ['club', 'residential', 'commercial']]

In [78]:
len(forecast_noise_levels)

6

In [80]:
noise_i

0

In [82]:
p_idx

0

In [73]:
C_L.coalitions

[['club'],
 ['residential'],
 ['club', 'residential'],
 ['commercial'],
 ['club', 'commercial'],
 ['residential', 'commercial'],
 ['club', 'residential', 'commercial']]